# Level 6: AI-Assisted Scientific Programming, Reproducibility, Testing, and Presentation

**Course:** ICS 2207 — Scientific Computing
**Project:** HydroSense-Kenya
**Objective:** Demonstrate responsible AI use, reproducible workflows, validation, and scientific communication.

---

## 1. AI-Assisted Programming Tasks

We used **Claude (Anthropic)** for **one focused, heavily-iterated support task**: generating, auditing, and expanding the automated test suite for `src/numerical_methods.py`. The work was carried out over five iterative prompts:

1. **Initial test scaffolding** — AI generated separate test files for the root-finding methods (`test_root_finding.py`) and the linear-system solver (`test_linear_systems.py`); we corrected the file naming and import paths to match our actual `tests/` folder structure.
2. **Import path correction** — after sharing our project folder structure, AI rewrote the relative import block (`sys.path.insert(...)`) so both files resolve `numerical_methods.py` correctly from `tests/`.
3. **Design clarification** — we questioned why test-input functions (e.g. `f_quadratic`, `f_cubic`) were defined inside the test files; AI explained these are mathematical inputs passed *into* the solvers, not redefinitions of the solvers themselves. We confirmed this was correct and made no changes.
4. **Gap analysis** — AI audited the existing tests and flagged missing edge cases: roots at bracket midpoints/endpoints, `ValueError`/`ZeroDivisionError` conditions, singular matrices, float coefficients, row-order independence, and cross-method agreement checks.
5. **Expanding coverage** — AI rewrote both test files to incorporate every identified gap, which we then reviewed line-by-line for correctness and relevance before accepting.

Full prompt-by-prompt detail is in [`AI_USE_LOG.md`](../AI_USE_LOG.md), printed in Section 6.2 below.

**Key principle:** AI was used as a productivity aid, not as an authority. Every output was reviewed, modified where necessary, and validated by running it against our own implementation in `src/numerical_methods.py`.

---
## 2. Automated Testing

We wrote **51 automated tests** across 4 test files using pytest:

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-8.3.5, pluggy-1.6.0 -- /usr/bin/python
cachedir: .pytest_cache
rootdir: /home/claude/HydroSense-Kenya/HydroSense-Kenya
configfile: pyproject.toml
plugins: anyio-4.13.0
collecting ... collected 51 items

tests/test_integration.py::test_rk4_linear_function PASSED               [  1%]
tests/test_integration.py::test_physical_constraint PASSED               [  3%]
tests/test_linear_systems.py::TestGaussianElimination::test_1x1_system PASSED [  5%]
tests/test_linear_systems.py::TestGaussianElimination::test_2x2_simple PASSED [  7%]
tests/test_linear_systems.py::TestGaussianElimination::test_3x3_standard PASSED [  9%]
tests/test_linear_systems.py::TestGaussianElimination::test_identity_matrix PASSED [ 11%]
tests/test_linear_systems.py::TestGaussianElimination::test_diagonal_matrix PASSED [ 13%]
tests/test_linear_systems.py::TestGaussianElimination::test_negative_rhs PASSED

### Test Coverage Summary

| Test File | Module Tested | Tests | Coverage |
|-----------|---------------|-------|----------|
| `test_root_finding.py` | `numerical_methods.py` — bisection, Newton-Raphson, secant | 32 | Convergence, accuracy, edge cases (midpoint/near-endpoint roots, same-sign brackets, zero derivative, flat secant), cross-method agreement |
| `test_linear_systems.py` | `numerical_methods.py` — Gaussian elimination | 15 | 1×1 to 4×4 systems, partial pivoting, singular matrices, negative/float coefficients, residual checks (Ax ≈ b), row-order independence |
| `test_integration.py` | `numerical_methods.py` — RK4 ODE integrator | 2 | Analytical linear ODE check (dS/dt = 2t), physical non-negativity constraint |
| `test_simulation.py` | `simulation.py` — `simulate_and_optimize` | 2 | Irrigation triggers when projected moisture < S_min; no irrigation when moisture stays sufficient |

> **Note:** during preparation of this notebook we discovered that `test_float_coefficients` (in `test_linear_systems.py`) had an incorrect expected solution hard-coded in its docstring and assertions (it claimed x=1.0, y=2.0 for a system whose correct analytical solution is x=1.75, y=1.75 — our `gaussian_elimination` implementation was already returning the *correct* value). We corrected the test's expected values; this is discussed further in Section 6 (Code Audit Preparation).

---
## 3. Reproducibility Checklist

In [ ]:
import os

checklist = [
    ('README.md', '../README.md'),
    ('requirements.txt', '../requirements.txt'),
    ('pyproject.toml', '../pyproject.toml'),
    ('AI_USE_LOG.md', '../AI_USE_LOG.md'),
    ('main.py (single-command pipeline)', '../main.py'),
    ('Raw data: weather_daily.csv', '../data/raw/weather_daily.csv'),
    ('Raw data: soil_sensor_data.csv', '../data/raw/soil_sensor_data.csv'),
    ('Raw data: crop_zone_parameters.csv', '../data/raw/crop_zone_parameters.csv'),
    ('Processed: cleaned_irrigation_dataset.csv', '../data/processed/cleaned_irrigation_dataset.csv'),
    ('Processed: level_2_results.csv', '../data/processed/level_2_results.csv'),
    ('src/data_cleaning.py', '../src/data_cleaning.py'),
    ('src/numerical_methods.py', '../src/numerical_methods.py'),
    ('src/simulation.py', '../src/simulation.py'),
    ('src/optimization.py', '../src/optimization.py'),
    ('src/visualization.py', '../src/visualization.py'),
    ('Level 1 notebook', '../notebooks/Level_1_Problem_Framing.ipynb'),
    ('Level 2 notebook', '../notebooks/Level_2_Vectorization_and_Error.ipynb'),
    ('Level 3 notebook', '../notebooks/Level_3_Numerical_Methods.ipynb'),
    ('Level 4 notebook', '../notebooks/Level_4_Data_Analysis_and_Visualization.ipynb'),
    ('Level 5 notebook', '../notebooks/Level_5_Simulation_and_Optimization.ipynb'),
    ('Level 6 notebook', '../notebooks/Level_6_Final_Integration.ipynb'),
    ('tests/test_root_finding.py', '../tests/test_root_finding.py'),
    ('tests/test_integration.py', '../tests/test_integration.py'),
    ('tests/test_linear_systems.py', '../tests/test_linear_systems.py'),
    ('tests/test_simulation.py', '../tests/test_simulation.py'),
    ('reports/final_scientific_report.pdf', '../reports/final_scientific_report.pdf'),
    ('reports/presentation_slides.pdf', '../reports/presentation_slides.pdf'),
]

print(f"{'Item':<45} {'Status'}")
print('=' * 55)
for name, path in checklist:
    exists = os.path.exists(path)
    print(f"{name:<45} {'PRESENT' if exists else 'MISSING'}")

Item                                          Status
README.md                                     PRESENT
requirements.txt                              PRESENT
pyproject.toml                                PRESENT
AI_USE_LOG.md                                 PRESENT
main.py (single-command pipeline)             PRESENT
Raw data: weather_daily.csv                   PRESENT
Raw data: soil_sensor_data.csv                PRESENT
Raw data: crop_zone_parameters.csv            PRESENT
Processed: cleaned_irrigation_dataset.csv     PRESENT
Processed: level_2_results.csv                PRESENT
src/data_cleaning.py                          PRESENT
src/numerical_methods.py                      PRESENT
src/simulation.py                             PRESENT
src/optimization.py                           PRESENT
src/visualization.py                          PRESENT
Level 1 notebook                              PRESENT
Level 2 notebook                              PRESENT
Level 3 notebook             

---
## 4. Full Pipeline Validation

`main.py` is the single-command, end-to-end pipeline for this project: it cleans the raw data (Level 1), computes the vectorized water-balance and ET equations (Level 2), and runs the irrigation optimizer (Level 5). Running it here, exactly as a fresh clone of this repository would, verifies reproducibility end-to-end.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'main.py'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

INITIALIZING HYDROSENSE-KENYA PIPELINE

[1/3] EXECUTING LEVEL 1: Data Cleaning & Imputation...
Saved: ../data/processed/cleaned_irrigation_dataset.csv
Shape: (90, 18)  (30 days × 3 zones)
Remaining NaNs: 0

[2/3] EXECUTING LEVEL 2: NumPy Vectorized Water Balance...
   -> Loading clean dataset...
   -> Applying C-level vectorized ET and Drainage equations...
   -> Level 2 simulation saved to: data/processed/level_2_results.csv

[3/3] EXECUTING LEVEL 3: Testing Continuous ODE Solvers...
   -> Loading numerical.py engines (Euler & RK4)...
   -> Successfully loaded 90 rows for ODE analysis.
   -> ODE plotting is reserved for notebooks/Level_3_Differential_Equations.ipynb

[3/3] EXECUTING LEVEL 5: Optimization Algorithm...
   -> Loading Level 2 results for optimization...
   -> Optimization Result: 0.00 mm total water required.

 PIPELINE COMPLETE! All systems nominal.



### 4.1 — Numerical Methods Spot Check

In addition to the full pipeline, we spot-check the three core root-finding methods on the textbook function $f(x) = x^2 - 4$ (root at $x=2$), the Gaussian elimination solver on a simple 2×2 system, and the Euler vs RK4 ODE integrators on the same linear ODE used in `tests/test_integration.py` ($dS/dt = 2t$, analytical solution $S(t) = t^2$).

In [ ]:
import sys
import numpy as np
sys.path.insert(0, '../src')

from numerical_methods import (
    bisection_method, newton_raphson_method, secant_method,
    gaussian_elimination, euler_integrate, rk4_integrate
)

# --- Root finding: f(x) = x^2 - 4, root at x = 2 ---
f  = lambda x: x**2 - 4
df = lambda x: 2*x

root_b, it_b = bisection_method(f, 0, 3)
root_n, it_n = newton_raphson_method(f, df, 3.0)
root_s, it_s = secant_method(f, 1.0, 3.0)

print("Root finding on f(x) = x^2 - 4 (expected root = 2.0):")
print(f"  Bisection:      root = {root_b:.6f}  ({it_b} iterations)")
print(f"  Newton-Raphson: root = {root_n:.6f}  ({it_n} iterations)")
print(f"  Secant:         root = {root_s:.6f}  ({it_s} iterations)")
print()

# --- Linear system: 2x + y = -5, x + 3y = -10  ->  x=-1, y=-3 ---
A = [[2, 1], [1, 3]]
b = [-5, -10]
x = gaussian_elimination(A, b)
print(f"Gaussian elimination on 2x+y=-5, x+3y=-10: x = {[round(v, 4) for v in x]} (expected [-1.0, -3.0])")
print()

# --- ODE integrators: dS/dt = 2t, S(0) = 0 -> analytical S(t) = t^2 ---
derivative = lambda t, S: 2 * t
t_array = np.linspace(0, 1, 11)
S_euler = euler_integrate(derivative, 0.0, t_array)
S_rk4   = rk4_integrate(derivative, 0.0, t_array)
analytical = t_array[-1] ** 2

print(f"ODE check (dS/dt = 2t), analytical S(1) = {analytical:.6f}:")
print(f"  Euler S(1) = {S_euler[-1]:.6f}  | error = {abs(S_euler[-1] - analytical):.2e}")
print(f"  RK4   S(1) = {S_rk4[-1]:.6f}  | error = {abs(S_rk4[-1] - analytical):.2e}")
print()
print("Full pipeline and spot checks validated successfully!")

Root finding on f(x) = x^2 - 4 (expected root = 2.0):
  Bisection:      root = 1.999998  (19 iterations)
  Newton-Raphson: root = 2.000000  (5 iterations)
  Secant:         root = 2.000000  (6 iterations)

Gaussian elimination on 2x+y=-5, x+3y=-10: x = [-1.0, -3.0] (expected [-1.0, -3.0])

ODE check (dS/dt = 2t), analytical S(1) = 1.000000:
  Euler S(1) = 0.900000  | error = 1.00e-01
  RK4   S(1) = 1.000000  | error = 0.00e+00

Full pipeline and spot checks validated successfully!


---
## 5. Project Statistics

In [ ]:
import glob

def count_lines(pattern):
    total = 0
    files = [f for f in glob.glob(pattern, recursive=True) if '__pycache__' not in f]
    for f in files:
        try:
            with open(f) as fh:
                total += sum(1 for _ in fh)
        except OSError:
            pass
    return total, len(files)

src_lines, src_files   = count_lines('../src/*.py')
test_lines, test_files = count_lines('../tests/test_*.py')

stats = [
    ('Source modules (src/)', f'{src_files} files, {src_lines} lines'),
    ('Test files (tests/)', f'{test_files} files, {test_lines} lines'),
    ('Notebooks', '6 notebooks (Levels 1-6)'),
    ('Datasets', '3 raw CSVs + 2 processed CSVs'),
    ('Numerical methods', '8 functions implemented from scratch (numerical_methods.py)'),
    ('Optimization / simulation', '2 functions (simulation.py, optimization.py)'),
    ('Automated tests', '51 tests across 4 files, 51/51 passing'),
    ('AI-assisted tasks', '1 task area, 5 iterative prompts (validated and reviewed)'),
]

print(f"{'Metric':<35} {'Value'}")
print('=' * 60)
for metric, value in stats:
    print(f'{metric:<35} {value}')

Metric                              Value
Source modules (src/)               5 files, 388 lines
Test files (tests/)                 4 files, 580 lines
Notebooks                           6 notebooks (Levels 1-6)
Datasets                            3 raw CSVs + 2 processed CSVs
Numerical methods                   8 functions implemented from scratch (numerical_methods.py)
Optimization / simulation           2 functions (simulation.py, optimization.py)
Automated tests                     51 tests across 4 files, 51/51 passing
AI-assisted tasks                   1 task area, 5 iterative prompts (validated and reviewed)


---
## 6. Code Audit Preparation

### 6.1 — Questions We Can Defend

1. **Why bisection over Newton-Raphson?** Bisection is guaranteed to converge once a valid sign-change bracket is found, at the cost of more iterations (19 vs 5 for $f(x)=x^2-4$ in Section 4). Newton-Raphson converges much faster but requires a derivative and can diverge or raise `ZeroDivisionError` for a bad starting point — which is why `newton_raphson_method` explicitly raises when $f'(x_0) = 0$ and our tests check for it.

2. **Why these specific data-cleaning decisions?** Each anomaly in `src/data_cleaning.py` was handled based on its likely cause: the 45.8 °C reading was replaced with the temperature median excluding itself (physically implausible for Kenya, single-point sensor error); `rainfall_mm` NaNs were filled with the median (right-skewed distribution, median is more robust than mean); `humidity_pct` NaNs were filled with the mean (roughly symmetric distribution); and Zone B/Zone C outliers (`soil_moisture_pct = 8.5%`, `tank_level_liters = 9900`) were replaced with zone-specific medians excluding the outlier, since both are isolated, physically implausible single readings rather than genuine trend changes.

3. **What happens when a sensor reports `CHECK`?** In `data_cleaning.py`, any Zone B row with `sensor_status == 'CHECK'` and `pump_flow_lpm == 0.0` has its flow rate replaced with the Zone B median (excluding zeros), and its status is relabelled `IMPUTED` — so downstream analysis can still distinguish originally-clean readings from sensor-fault corrections if needed.

4. **Why a greedy / Model Predictive Control approach for irrigation, rather than a global optimizer?** `simulate_and_optimize` / `optimize_irrigation_schedule` make a one-day-ahead projection and apply exactly enough water to reach `S_target` only when projected moisture would fall below `S_min`. This is transparent, easy to verify by hand, and guaranteed to keep moisture within bounds — at the cost of being reactive rather than forecast-based. A multi-day, forecast-aware optimizer is listed as future work.

5. **How do you know the numerical methods are correct?** Every method in `numerical_methods.py` is checked against a known analytical solution: bisection/Newton-Raphson/secant all agree on $x=2$ for $f(x)=x^2-4$ (and cross-validate against each other in `test_root_finding.py`); `gaussian_elimination` is checked via residuals ($Ax \approx b$) and against `np.linalg.solve`-equivalent hand-solved systems; `rk4_integrate` reproduces the exact analytical solution of $dS/dt=2t$. All 51 tests pass.

6. **Why do `simulation.py` and `optimization.py` contain near-identical functions (`simulate_and_optimize` and `optimize_irrigation_schedule`)?** This is a known duplication from iterative development: the optimization logic was first prototyped in `simulation.py` (and is still imported by `tests/test_simulation.py` and `main.py`'s ODE step), then refactored into `optimization.py` as the canonical version used by `main.py`'s Level 5 step. Before final submission this should be consolidated into a single source of truth to avoid the two implementations drifting apart.

7. **What did your own audit find, and how was it handled?** While preparing this notebook we ran the full test suite and found `test_float_coefficients` (`test_linear_systems.py`) asserted an incorrect expected solution (x=1.0, y=2.0) for the system $1.5x+2.5y=7,\ 0.5x+1.5y=3.5$, whose correct analytical solution is x=1.75, y=1.75. Our `gaussian_elimination` implementation was already returning the correct value — the bug was in the test's hard-coded expectation, not the source code. We corrected the expected values (Section 2), re-ran the suite, and confirmed 51/51 tests now pass. This is recorded here as evidence that our test suite is independently audited, not just trusted at face value.

8. **What are the known limitations / future work items?** (a) `trapezoidal_rule` and `simpsons_rule` are implemented in `numerical_methods.py` but currently have no dedicated tests in `tests/` — this is a coverage gap to close. (b) `src/visualization.py` exists in the repository structure but is currently empty; all plotting is done inline in the Level 1–5 notebooks. (c) The `simulation.py` / `optimization.py` duplication noted in Q6 should be resolved.

---
*End of Level 6 — AI-Assisted Scientific Programming, Reproducibility, Testing, and Presentation.*

*End of HydroSense-Kenya Capstone Project.*